# 03 · 原片精改三段式管线（多角色定向换脸 + 本尊音色中文克隆配音 + 说话人精准口型同步）

- **核心原则**：**100% 保留原片构图、清宫龙袍/朝服、光影、背景配乐与演员动作，不重新生成画面！**
- **推荐算力**：**Colab L4 GPU**（约 `1.7 CU/h`，极省算力）或 **A100 GPU**，全片 `0:00 – 4:11` (`shot01`–`shot21`) 预计 **8–15 分钟** 跑完并自动释放 GPU。
- **依赖隔离**：自动通过 `uv` 建立独立 `Python 3.10` 虚拟环境，彻底规避 Colab `Python 3.13` 轮子冲突。
- **成品输出**：`/MyDrive/AI_Video/projects/<PROJECT_ID>/outputs/final_cut/02_yongzheng_ep12_huang_shot01_21_replace.mp4`

In [ ]:
#@title 1. 挂载 5TB Google Drive、拉取最新仓库并安装独立 Python 3.10 GPU 运行环境
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}
PROJECT_ID = "02_yongzheng_ep12_huang" #@param {type:"string"}

import os, shutil, subprocess
from pathlib import Path

subprocess.run(["nvidia-smi"], check=False)
from google.colab import drive
drive.mount('/content/drive')

if not Path(REPO_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# 同步最新 dubbing_plan_shot01_21.json 到 5TB Drive 项目目录
PROJECT_DIR = Path(f"/content/drive/MyDrive/AI_Video/projects/{PROJECT_ID}")
plan_src = Path(REPO_DIR) / "projects" / PROJECT_ID / "dubbing_plan_shot01_21.json"
if plan_src.exists():
    shutil.copy2(plan_src, PROJECT_DIR / "dubbing_plan_shot01_21.json")
    print(f"[同步计划] 已更新 {PROJECT_DIR / 'dubbing_plan_shot01_21.json'}")

# 安装 uv 并建立隔离 Python 3.10 环境（规避 Colab Python 3.13 依赖冲突）
subprocess.run(["pip", "install", "-q", "uv"], check=True)
VENV_DIR = Path("/content/venv_replace")
VENV_PY = str(VENV_DIR / "bin" / "python")
if not Path(VENV_PY).exists():
    subprocess.run(["uv", "venv", str(VENV_DIR), "--python", "3.10"], check=True)

# 安装 PyTorch CUDA + 人脸聚类换脸 + GFPGAN + Whisper + Demucs + Chatterbox Multilingual
subprocess.run([
    "uv", "pip", "install", "--python", VENV_PY,
    "torch==2.6.0", "torchaudio==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124"
], check=True)
subprocess.run([
    "uv", "pip", "install", "--python", VENV_PY,
    "numpy<2.0", "scipy", "librosa", "soundfile", "pillow", "scikit-learn",
    "opencv-python-headless", "insightface",
    "gfpgan", "basicsr", "facexlib", "realesrgan",
    "faster-whisper", "demucs", "chatterbox-tts"
], check=True)
# 确保移除 insightface 带入的 CPU 版 onnxruntime，强制使用 CUDA 12 GPU 版 onnxruntime-gpu
subprocess.run(["uv", "pip", "uninstall", "--python", VENV_PY, "onnxruntime", "onnxruntime-gpu"], check=False)
subprocess.run(["uv", "pip", "install", "--python", VENV_PY, "onnxruntime-gpu"], check=True)

# 兼容 torchvision>=0.17 修复 basicsr functional_tensor 导入
subprocess.run([VENV_PY, "-c", """
import pathlib, site
for sp in site.getsitepackages():
    deg = pathlib.Path(sp) / 'basicsr' / 'data' / 'degradations.py'
    if deg.exists():
        txt = deg.read_text(encoding='utf-8')
        old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        new = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if old in txt:
            deg.write_text(txt.replace(old, new), encoding='utf-8')
            print('[修复] basicsr functional_tensor 兼容补丁已应用')
"""], check=True)

# 拉取 Wav2Lip 模型定义（仅用其网络结构 models/wav2lip.py，权重缓存于 5TB Drive）
if not Path("/content/Wav2Lip").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Rudrabha/Wav2Lip.git", "/content/Wav2Lip"], check=True)

print("\n[环境就绪] Python 3.10 独立 GPU 环境与依赖安装完成！")

In [ ]:
#@title 2. Stage 1：原片裁剪 (0:00–4:11) + Demucs 保留背景配乐 + Whisper 逐句对齐 + 人脸聚类核验
WORK_DIR = "/content/work_replace" #@param {type:"string"}

import subprocess
from pathlib import Path
from IPython.display import Image as IPyImage, display

def run_stream(cmd):
    import glob, os
    env = os.environ.copy()
    nv_libs = glob.glob('/content/venv_replace/lib/python3.10/site-packages/nvidia/*/lib')
    env['LD_LIBRARY_PATH'] = ':'.join(nv_libs + [env.get('LD_LIBRARY_PATH', '')])
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f'Command failed with exit code {rc}: {cmd}')

run_stream([
    VENV_PY, "-u", f"{REPO_DIR}/scripts/video_replace_pipeline.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "analyze",
])

clusters_img = Path(WORK_DIR) / "actor_clusters.jpg"
if clusters_img.exists():
    print("\n=== 演员人脸自动聚类对照表（如某个 Cluster 映射有误，可在下一个单元格修改 actor_face_map.json）===")
    display(IPyImage(filename=str(clusters_img)))

In [ ]:
#@title 3. Stage 2：拉取 4 位主角纯净干音 + 跨语种中文克隆配音 + 混入原片配乐环境声
TTS_ENGINE = "chatterbox" #@param ["chatterbox"]

import subprocess
from pathlib import Path
from IPython.display import Audio, display

run_stream([
    VENV_PY, "-u", f"{REPO_DIR}/scripts/video_replace_pipeline.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "dub",
    "--tts-engine", TTS_ENGINE,
])

# 内嵌试听前两句克隆中文配音 + 最终混音前 20 秒
for sample in sorted((Path(WORK_DIR) / "tts_lines").glob("*_fit.wav"))[:3]:
    print(f"试听: {sample.name}")
    display(Audio(filename=str(sample)))

In [ ]:
#@title 4. Stage 3 & 4：原片逐帧多角色定向换脸 + 说话人精准口型同步 + 封装成片 + 自动释放 GPU
SKIP_LIPSYNC = False #@param {type:"boolean"}
AUTO_DISCONNECT = True #@param {type:"boolean"}

import subprocess, time
from pathlib import Path

run_stream([
    VENV_PY, "-u", f"{REPO_DIR}/scripts/video_replace_pipeline.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "faceswap",
])

lipsync_cmd = [
    VENV_PY, "-u", f"{REPO_DIR}/scripts/video_replace_pipeline.py",
    "--project-dir", str(PROJECT_DIR),
    "--work-dir", WORK_DIR,
    "--stage", "lipsync",
    "--lipsync-repo", "/content/Wav2Lip",
]
if SKIP_LIPSYNC:
    lipsync_cmd.append("--skip-lipsync")
run_stream(lipsync_cmd)

# 把中间审计产物 (actor_clusters.jpg / aligned_dub_timeline.json / orig_transcript.json) 也备份到项目 outputs/replace_debug/
debug_out = PROJECT_DIR / "outputs" / "replace_debug"
debug_out.mkdir(parents=True, exist_ok=True)
for fname in ("actor_clusters.jpg", "actor_face_map.json", "aligned_dub_timeline.json", "orig_transcript.json"):
    p = Path(WORK_DIR) / fname
    if p.exists():
        subprocess.run(["cp", "-f", str(p), str(debug_out / fname)], check=False)

subprocess.run(["sync"], check=False)
time.sleep(15)

final_mp4 = PROJECT_DIR / "outputs" / "final_cut" / "02_yongzheng_ep12_huang_shot01_21_replace.mp4"
print(f"\n=== 全片原片精改完成！成片位置: {final_mp4} ===")

if AUTO_DISCONNECT and final_mp4.exists() and final_mp4.stat().st_size > 1_000_000:
    print("[自动止损] 成片已安全落盘 5TB Google Drive，自动释放 GPU 运行时！")
    from google.colab import runtime
    runtime.unassign()